# SetFit for Text Classification

In this notebook, we'll learn how to do few-shot text classification with SetFit.

## Setup

If you're running this Notebook on Colab or some other cloud platform, you will need to install the `setfit` library. Uncomment the following cell and run it:

In [ ]:
# %pip install setfit

To be able to share your model with the community, there are a few more steps to follow.

First, you have to store your authentication token from the Hugging Face Hub (sign up [here](https://huggingface.co/join) if you haven't already!). To do so, execute the following cell and input an [access token](https://huggingface.co/docs/hub/security-tokens) associated with your account:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Then you need to install Git-LFS, which you can do by uncommenting and running following command:

In [ ]:
# !apt install git-lfs

Finally, you may need to configue Git on your system by providing details about who you are:

In [1]:
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"

This notebook is designed to work with any multiclass [text classification dataset](https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads) and pretrained [Sentence Transformer](https://huggingface.co/models?library=sentence-transformers&sort=downloads) on the Hub. Change the values below to try a different dataset / model!

In [2]:
dataset_id = "stanfordnlp/sst2"
model_id = "sentence-transformers/paraphrase-mpnet-base-v2"

## Loading and sampling the dataset

We will use the 🤗 Datasets library to download the data, which can be done as follows:

In [3]:
from datasets import load_dataset

dataset = load_dataset(dataset_id)
dataset

Using custom data configuration default
Reusing dataset sst2 (/home/lewis/.cache/huggingface/datasets/sst2/default/2.0.0/9896208a8d85db057ac50c72282bcb8fe755accc671a57dd8059d4e130961ed5)


  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

Most datasets on the Hub have many more labeled examples than those one encounters in few-shot settings. To simulate the effect of training on a limited number of examples, let's subsample the training set to have 8 labeled examples per class:

In [4]:
from setfit import sample_dataset

train_dataset = sample_dataset(dataset["train"])
train_dataset

Loading cached shuffled indices for dataset at /home/lewis/.cache/huggingface/datasets/sst2/default/2.0.0/9896208a8d85db057ac50c72282bcb8fe755accc671a57dd8059d4e130961ed5/cache-2459d1a782cafb86.arrow
Parameter 'function'=<function sample_dataset.<locals>.<lambda> at 0x7faf0a689630> of the transform datasets.arrow_dataset.Dataset.filter@2.0.1 couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


  0%|          | 0/68 [00:00<?, ?ba/s]

  0%|          | 0/68 [00:00<?, ?ba/s]

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 16
})

Here we have 16 total examples to train with since the `sst2` dataset has two classes (positive and negative). For evaluation, we'll use the validation split, since the test split of `sst2` is unlabeled:

In [5]:
eval_dataset = dataset["validation"] 

Okay, now we have the dataset, let's load and train a model!

## Fine-tuning the model

To train a SetFit model, the first thing to do is download a pretrained checkpoint from the Hub. We can do so by using the `from_pretrained()` method associated with the `SetFitModel` class:

In [39]:
from setfit import SetFitModel

model = SetFitModel.from_pretrained(model_id)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


The model now has a pretrained Sentence Transformer body and a logistic regression head. Configure contrastive training with `TrainingArguments`, then pass the arguments and datasets to `Trainer`.

In [ ]:
from sentence_transformers.losses import CosineSimilarityLoss

from setfit import Trainer, TrainingArguments

args = TrainingArguments(loss=CosineSimilarityLoss, num_iterations=20)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    column_mapping={"sentence": "text", "label": "label"},
)

* `loss` sets the contrastive loss for training the Sentence Transformer body.
* `num_iterations` controls how many positive and negative pairs are sampled per training example.
* `column_mapping` tells the trainer which dataset columns contain the text and labels.

Now that we've created a trainer, we can train it!

In [ ]:
trainer.train()

The final step is to compute the model's performance using the `evaluate()` method:

In [ ]:
metrics = trainer.evaluate()
metrics

And once the model is trained, you can push it to the Hub:

In [ ]:
trainer.push_to_hub("my-awesome-setfit-model")

You can now share this model with all your friends, family, favorite pets: they can all load it with the identifier `your-username/the-name-you-picked` so for instance:

In [11]:
from setfit import SetFitModel

model = SetFitModel.from_pretrained("lewtun/my-awesome-setfit-model")

# Run inference
preds = model(["i loved the spiderman movie!", "pineapple on pizza is the worst 🤮"])
preds   

/home/lewis/miniconda3/envs/setfit/lib/python3.10/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator LogisticRegression from version 1.1.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


array([1, 0])

## Fine-tuning with a pure PyTorch model

`setfit` also provides a pure PyTorch implementation of `SetFitModel`, where the head is a dense layer instead of a classifier from `scikit-learn`. This allows one to do backprop end-to-end and have more fine-grained control over the training process.

To use the PyTorch model, we load a pretrained model with `use_differentiable_head=True` and specify the number of classes to include in the head:

In [48]:
from setfit import SetFitModel

num_classes = len(train_dataset.unique("label"))
model = SetFitModel.from_pretrained(model_id, use_differentiable_head=True, head_params={"out_features": num_classes})

Loading cached processed dataset at /home/lewis/.cache/huggingface/datasets/sst2/default/2.0.0/9896208a8d85db057ac50c72282bcb8fe755accc671a57dd8059d4e130961ed5/cache-f8c1ea1d1209fce8.arrow
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


For the PyTorch head, set the epochs and learning rates for each training phase. Here the body trains for one epoch at `1e-5`, then the head trains for 50 epochs at `1e-2`. The default `end_to_end=False` keeps the body frozen during head training.

In [ ]:
args = TrainingArguments(
    loss=CosineSimilarityLoss,
    num_iterations=20,
    num_epochs=(1, 50),
    body_learning_rate=1e-5,
    head_learning_rate=1e-2,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    column_mapping={"sentence": "text", "label": "label"},
)

Run both training phases with a single call:

In [ ]:
trainer.train()

Evaluate the trained PyTorch model on the same validation set:

In [ ]:
trainer.evaluate()